# Export Streaming & Invoice Template Notebook

This notebook demonstrates server-side pagination and streaming for large CSV exports in FastAPI, and how to import a simple invoice template/design from an external repo into a frontend preview.

## 1) Import Required Libraries

We'll import FastAPI testing utilities, SQLAlchemy ORM session mocks, StreamingResponse, and basic helpers. In a real app these imports live in your FastAPI app files; here they are for demonstration and testing.

In [ ]:
```python
# core libs for demonstration
from fastapi import FastAPI, Depends, Query
from fastapi.responses import StreamingResponse
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime
from sqlalchemy.orm import sessionmaker, declarative_base
import csv, io, json
from datetime import datetime

# frontend preview helper
from IPython.display import HTML
```

## 2) Set Up Sample Data and Configuration

We'll create an in-memory SQLite DB and populate invoice-like rows so we can exercise pagination/streaming without a real Postgres instance.

In [ ]:
```python
Base = declarative_base()
engine = create_engine('sqlite:///:memory:')
SessionLocal = sessionmaker(bind=engine)

class Invoice(Base):
    __tablename__ = 'invoices'
    id = Column(Integer, primary_key=True)
    invoice_number = Column(String)
    society_id = Column(Integer)
    unit_id = Column(Integer)
    billing_month = Column(String)
    net_amount = Column(Float)
    status = Column(String)
    created_at = Column(DateTime, default=datetime.utcnow)

Base.metadata.create_all(engine)

# populate sample data
s = SessionLocal()
for i in range(1, 5001):
    s.add(Invoice(invoice_number=f"INV-2026-{i:04d}", society_id=1, unit_id=(i%100)+1, billing_month='2026-05', net_amount=100.0+i%10, status='pending' if i%3 else 'paid'))

s.commit()
print('seeded')
```

## 3) Implement Server-Side Pagination for Large Exports

This example shows a simple paginated endpoint pattern that returns chunks of CSV rows as JSON or CSV files for a single page. Use `limit` and `offset` query parameters to control server-side paging.

In [ ]:
```python
# paginated fetch function
from fastapi import APIRouter
router = APIRouter()

@router.get('/export/page')
def export_page(society_id: int = Query(...), limit: int = Query(500), offset: int = Query(0)):
    db = SessionLocal()
    query = db.query(Invoice).filter(Invoice.society_id == society_id).order_by(Invoice.id)
    rows = query.offset(offset).limit(limit).all()
    # build CSV for this page
    out = io.StringIO()
    writer = csv.writer(out)
    writer.writerow(['id','invoice_number','unit_id','billing_month','net_amount','status','created_at'])
    for r in rows:
        writer.writerow([r.id, r.invoice_number, r.unit_id, r.billing_month, r.net_amount, r.status, r.created_at.isoformat()])
    return Response(content=out.getvalue(), media_type='text/csv')
```

## 4) Implement Streaming Export Endpoint

Use a generator to yield CSV rows in batches without loading all invoices into memory. This endpoint will stream progressively to the client.

In [ ]:
```python
@router.get('/export/stream')
def export_stream(society_id: int = Query(...), batch_size: int = Query(500)):
    def row_generator():
        db = SessionLocal()
        offset = 0
        # header
        out = io.StringIO()
        csv.writer(out).writerow(['id','invoice_number','unit_id','billing_month','net_amount','status','created_at'])
        yield out.getvalue()
        while True:
            rows = db.query(Invoice).filter(Invoice.society_id == society_id).order_by(Invoice.id).offset(offset).limit(batch_size).all()
            if not rows:
                break
            for r in rows:
                out = io.StringIO()
                csv.writer(out).writerow([r.id, r.invoice_number, r.unit_id, r.billing_month, r.net_amount, r.status, r.created_at.isoformat()])
                yield out.getvalue()
            offset += batch_size
    return StreamingResponse(row_generator(), media_type='text/csv', headers={'Content-Disposition':'attachment; filename=invoices_stream.csv'})
```

## 5) Fetch and Integrate External Invoice UI Assets

We'll load the HTML invoice snippet from the external Flask template and adapt it for the frontend invoice preview (the preview opened by the frontend uses the backend `/invoices/{id}/export` HTML). We'll demonstrate extraction of simple header/footer and CSS from the external repo's `user_bill_layout.css` and `userbillpage.html`.

In [ ]:
```python
# read external HTML template and CSS (from local repo)
html_path = r'external-repos/Society-Management/App/templates/user/userbillpage.html'
css_path = r'external-repos/Society-Management/App/static/css/user_bill_layout.css'

with open(html_path, 'r', encoding='utf-8') as f:
    external_html = f.read()
with open(css_path, 'r', encoding='utf-8') as f:
    external_css = f.read()

# show a small excerpt
external_html[:1000]
```

## 6) Apply Imported Invoice Template Design

We'll build a small HTML preview by combining header/footer and CSS extracted from the external template and render it inline in the notebook to verify appearance.

In [ ]:
```python
# create a small invoice rendering using a single invoice from DB
db = SessionLocal()
inv = db.query(Invoice).first()
line_items_html = '<tr><td>Maintenance</td><td>1</td><td>100.00</td><td>100.00</td></tr>'
preview_html = f"""
<html>
<head>
  <style>{external_css}
  table{{width:100%;border-collapse:collapse}} td,th{{border:1px solid #ddd;padding:8px}}</style>
</head>
<body>
  <div class='currBillContainer'>
    <h1>Invoice {inv.invoice_number}</h1>
    <div>Date: {inv.created_at.date()}</div>
    <table>
      <thead><tr><th>Category</th><th>Qty</th><th>Rate</th><th>Total</th></tr></thead>
      <tbody>
        {line_items_html}
        <tr class='table-active'><td>Total</td><td></td><td></td><td>{inv.net_amount:.2f}</td></tr>
      </tbody>
    </table>
  </div>
</body>
</html>
"""

HTML(preview_html)
```

## 7) Validate Export Performance and UI Preview

We test the streaming endpoint locally (in-app) by iterating the generator and measuring memory usage and time. For the UI preview we rendered the HTML above; in a real app the backend `/invoices/{id}/export` should use the same HTML/CSS assembly.